# CopyLane 판정 인코더 v10 — dev 를 v11 과 같게 · 합성 문구 × 의결서 「이유」 구역 실험 (박수진, 2026-10-07)

> 골든 `529c970556d7` · 8,383행 (재동결 10-05) · 확정 8종 + 편입 대기 1 · 저장 9칸 (D-321)
> 🔴 **dev 만 본다. 이 노트북에는 test 채점 셀이 없다** (D-175 — test 는 최종 후보 1회 · 팀장 판정 뒤).

## 왜 만들었나

후보 A(v8.2)와 v11(소성민)은 같은 골든으로 학습했지만 **dev 를 떼는 규칙이 달라** 겹치는 dev 행이 49줄뿐이었다. 그래서 dev 수치로는 둘을 견줄 수 없었다.
이 노트북은 **dev 를 v11 규칙 그대로**(원천별 묶음 15% + 6종 보충 · 시드 42 → 749줄) 떼고, 그 위에서 A 의 학습 방식으로 네 가지를 돌린다.

| 실험 | 합성(주입) 문구 | 의결서 「이유」 구역 | 뜻 |
|---|---|---|---|
| `합성O·이유O` (**기준**) | 넣음 | 넣음 | A 와 같은 재료. v11 과도 거의 같다(낱말 행을 뺀 것 · 학습 루프 구현이 다르다) |
| `합성X·이유O` | 뺌 | 넣음 | 합성 문구가 실제로 도움이 되는가 |
| `합성O·이유X` | 넣음 | 뺌 | 「이유」 구역이 거짓_과장 과발화의 원인인가 |
| `합성X·이유X` | 뺌 | 뺌 | 둘 다 뺀 실제 광고 문구만 |

네 실험 × 시드 3개 = **학습 12번**. 그 밖의 설정은 전부 같다 (KcBERT · lr 2e-5 · 6 epoch · 조기 종료 2 · 배치 32 · focal γ 1.5 · pos_weight √ 상한 5 · 낱말 행 제외 · 조건 M 제외 · 유형 없는 위반 제외 · D · L 음성).

## dev 를 읽는 두 가지 방식

dev **행은 하나**(749줄)다. 다만 어떤 행을 채점에 넣느냐가 A 와 v11 이 달랐다. 표에는 **둘 다** 찍는다.

| 방식 | 채점 행 | 음성으로 치는 것 |
|---|---|---|
| **v11 방식** (기본 · `SCORE_VIEW = "v11"`) | 749줄 전부 (그중 「이유」 구역 356줄) | 조건 M · 유형 없는 위반 · D · L |
| A 방식 | 「이유」 구역 · 조건 M · 유형 없는 위반을 뺀 줄 (325줄) | D · L |

회차(조기 종료) · 시드 · 문턱은 `SCORE_VIEW` 로 고른다. 기본은 v11 방식이다 — 요청이 「A 의 dev 규칙을 v11 과 같게」였기 때문이다.

## 이 dev 로 알 수 있는 것 · 없는 것

- 알 수 있다: 위반 문구를 놓치지 않는가 · 유형별 PR-AUC · 거짓_과장이 얼마나 자주 울리는가.
- **알 수 없다: 적법한 수정문구를 통과시키는가.** dev 의 음성은 D 81 · L 5 뿐이고, D 는 대부분 「주의사항 문구」다. 해설서 수정문구(`gf:`)는 0줄이다 (오탐 진단 10-06). 이 실험의 dev 결론은 위반 쪽까지다.
- dev 양성이 적은 유형은 수치가 흔들린다: 후기_체험기_기만 14 · 부당_비교광고 10 · 비방광고 25 (30건 미만 = 측정 불가 · D-40).

## 실행 순서

1. 런타임 유형 **GPU(T4)** → 위에서부터 끝까지 실행. `golden.jsonl` 을 올린다. (선택) `dev_probs_v11_kcbert_20261006-1013.csv` 를 같이 올리면 **v11 줄이 같은 표에** 붙는다.
2. 시드 하나가 끝날 때마다 Drive `copylane_v10/` 에 기록한다 — 런타임이 끊겨도 다시 실행하면 **끝난 시드는 건너뛴다**.
3. 결과 표는 섹션 7. 실험마다 dev 행별 확률 CSV · 문턱 zip · 모델 zip 이 Drive 에 남는다 (**저장소에는 올리지 않는다** · D-249).

## 1. 환경 설정 + 파일 업로드

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

from google.colab import files as colab_files
if not os.path.exists("golden.jsonl"):
    print("① golden.jsonl 선택")
    colab_files.upload()
if not any(f.startswith("dev_probs_v11") and f.endswith(".csv") for f in os.listdir(".")):
    print("② (선택) dev_probs_v11_kcbert_20261006-1013.csv — v11 줄을 같은 표에 붙이려면 올린다. 없으면 「취소」")
    try:
        colab_files.upload()
    except Exception as e:
        print("건너뜀:", e)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")
V11_CSV = next((f for f in sorted(os.listdir(".")) if f.startswith("dev_probs_v11") and f.endswith(".csv")), None)
print("v11 dev 확률:", V11_CSV or "없음 (v11 줄 없이 간다)")

## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `RUNS` | 네 실험 전부 | 돌릴 실험. **기준(`합성O·이유O`)은 빼지 않는다** — 다른 실험을 기준에 견준다 |
| `SCORE_VIEW` | `"v11"` | 회차 · 시드 · 문턱을 고르는 dev 채점 방식. `"A"` 로 바꾸면 A 방식으로 고른다(다시 학습한다 · 결과는 다른 폴더) |
| `SAVE_MODEL_ZIP` | `True` | 실험마다 가장 좋은 시드의 모델 zip(약 420MB)을 Drive 에 둔다. Drive 가 모자라면 `False` |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `GATE` | 아래 값 | 🔒 **결과를 보기 전에만 고친다** (기준을 먼저 정한다) |

In [ ]:
RUN_DEFS = {                      # 이름 → 학습에 넣는 재료
    "합성O·이유O": {"inj": True,  "reason": True},     # 기준 — A 와 같은 재료
    "합성X·이유O": {"inj": False, "reason": True},
    "합성O·이유X": {"inj": True,  "reason": False},
    "합성X·이유X": {"inj": False, "reason": False},
}
RUNS = ["합성O·이유O", "합성X·이유O", "합성O·이유X", "합성X·이유X"]
BASE_RUN = "합성O·이유O"
SCORE_VIEW = "v11"                # "v11" = dev 749줄 전부(M · 유형 없는 위반 · D · L 음성) / "A" = 이유 구역 · M · 유형 없는 위반을 뺀 줄
SAVE_MODEL_ZIP = True
SMOKE = False

# ── 학습 설정 — A(v8.2) · v11 과 같다. 실험끼리 견주려면 바꾸지 않는다
MODEL_NAME = "beomi/kcbert-base"
SEEDS = [42, 43, 44]
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY, MAX_LEN = 0.01, 128
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 — 뺀 판이 후보 A 다 (팀장 동의 10-06)
DEV_FRAC, DEV_SEED = 0.15, 42     # v11 과 같다 — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)

# ── 🔒 사전 고정 기준 — 기준 실행(BASE_RUN) 대비. 결과를 보기 전에만 고친다
GATE = {
    "prauc_drop": 0.02,           # B  : 6종 dev macro PR-AUC(시드 평균) 하락 허용폭
    "recall_drop": 0.02,          # G1 : 위반 재현율(유형 있는 위반 · 8종 중 하나라도) 하락 허용폭
    "type_recall_drop": 0.05,     # G2 : 유형별 재현율 하락 허용폭 (dev 양성 30건 이상인 유형만 · D-40)
    "neg_flag_up": 1,             # G3 : 음성(D · L) 오판정이 늘어도 되는 줄 수
    "adopt_prauc": 0.008,         # 채택 후보 ① : PR-AUC 시드 평균이 이만큼 이상 오른다
    "adopt_neg_down": 3,          # 채택 후보 ② : 음성(D · L) 오판정이 이 줄 수 이상 준다
}

assert SCORE_VIEW in ("v11", "A"), SCORE_VIEW
assert BASE_RUN in RUNS and all(r in RUN_DEFS for r in RUNS), "RUNS 에 기준 실행이 있어야 한다"
RUNS = [BASE_RUN] + [r for r in RUNS if r != BASE_RUN]        # 기준을 먼저 돌린다
if SMOKE:
    SEEDS, MAX_EPOCHS = [42], 1
NOTEBOOK = "v10"
def run_tag(name):
    return f"copylane-encoder-kcbert-v10-{name.replace('·', '-')}" + ("" if SCORE_VIEW == "v11" else "-선택A")

OUT_ROOT = "/content/drive/MyDrive/copylane_v10" + ("" if SCORE_VIEW == "v11" else "_선택A") + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_v10_out"
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("🟡 Drive 연결 안 됨 — 결과를 런타임에만 둔다(런타임이 끊기면 사라진다):", e)
print("실험:", RUNS, "· 시드", SEEDS, "· 선택 방식", SCORE_VIEW, "· 결과 폴더", OUT_ROOT)

## 3. 데이터 + dev 분리 — **v11 규칙 그대로**

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # 조건 M · D · L 은 라벨이 적혀 있어도 양성이 아니다 (D-296 개정 2) — target 이 학습 · 채점의 정답이다
            "target": [] if cond in ("M", "D", "L") else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit")}

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5")}   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")

## 4. 학습셋 — 실험마다 재료만 다르다

dev 묶음을 뺀 train 에서 순서대로:

1. dev 와 같은 문구 제거 → 2. 조건 M 제외 (D-296 개정 2) → 3. 유형 없는 위반(C · A · B 인데 유형이 없다) 제외 → 4. 낱말 행 제외
5. **실험 스위치** — 합성(주입) 행 · 「이유」 구역 행을 넣거나 뺀다
6. 같은 문구 병합 (라벨은 합집합 · 이유/비이유는 따로)

D · L 은 음성으로 남는다. dev 묶음에서 나온 합성 행은 원본이 dev 에 있으므로 어느 실험에서도 학습하지 않는다.

In [ ]:
base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

def build_train(inj, reason):
    pool = [r for r in pool0 if (inj or not is_inj(r)) and (reason or not is_reason(r))]
    before, merged = len(pool), {}
    for r in pool:                                   # 같은 문구 병합 — 라벨 합집합 (A 와 같은 방식)
        k = (norm_text(r["text"]), is_reason(r))
        if k in merged:
            merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
        else:
            merged[k] = dict(r, target=list(r["target"]))
    rows = list(merged.values())
    if SMOKE:
        random.Random(0).shuffle(rows); rows = rows[:300]
    return rows, before

TRAIN, TRAIN_STATS = {}, {}
for name, cfg in RUN_DEFS.items():
    rows, before = build_train(cfg["inj"], cfg["reason"])
    TRAIN[name] = rows
    TRAIN_STATS[name] = {"병합 전": before, "학습 행": len(rows), "합성": sum(map(is_inj, rows)), "이유 구역": sum(map(is_reason, rows)),
                         "음성": sum(1 for r in rows if not r["target"]), **{l: sum(l in r["target"] for r in rows) for l in LABEL_LIST}}
_df = pd.DataFrame(TRAIN_STATS).T
print("\n■ 실험별 학습셋 (행 수 · 유형별 양성)" + ("   (SMOKE — 300줄로 줄였다)" if SMOKE else ""))
display(_df)
EXPECT_TRAIN = {"합성O·이유O": 3964, "합성X·이유O": 3306, "합성O·이유X": 2547, "합성X·이유X": 1889}   # 재동결 10-05 판 · 로컬 대조 10-07
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _bad = {n: (TRAIN_STATS[n]["학습 행"], EXPECT_TRAIN[n]) for n in RUN_DEFS if TRAIN_STATS[n]["학습 행"] != EXPECT_TRAIN[n]}
    print("✅ 학습 행 수가 기대값과 같다 (3,964 · 3,306 · 2,547 · 1,889)" if not _bad else f"🟡 학습 행 수가 기대값과 다르다 (받은, 기대) {_bad} — 학습셋 코드를 건드렸는지 본다")
_base, _nor = TRAIN_STATS["합성O·이유O"], TRAIN_STATS["합성O·이유X"]
print(f"참고 — 거짓_과장 양성 {_base['거짓_과장']:,}줄 중 「이유」 구역이 {_base['거짓_과장'] - _nor['거짓_과장']:,}줄이다. "
      "「이유」를 빼면 거짓_과장 학습 양성이 그만큼 준다.")
for name in RUNS:
    assert not (set(r["id"] for r in TRAIN[name]) & set(r["id"] for r in dev_rows)), "학습 행에 dev 행이 섞였다"
    assert not ({norm_text(r["text"]) for r in TRAIN[name]} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"

## 5. 손실 · 지표 — A(v8.2)와 같다

- Focal Loss(γ 1.5) · α = pos_weight = √(음성/양성) 을 1 ~ 5 로 자른 값. **실험마다 그 학습셋으로 다시 계산한다.**
- 회차(조기 종료) · 시드 = **6종 dev macro PR-AUC** 가 가장 높은 것 (D-321 결정 4 — 8종으로 넓히지 않는다).
- 문턱 = 확정 8종마다 dev F1 최대 (격자 0.1 ~ 0.9 · 0.025 · dev 양성 10건 미만이면 0.5). 편입 대기 칸은 1.01(꺼 둠).

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding, set_seed)
from sklearn.metrics import average_precision_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def to_ds(rows):
    ds = Dataset.from_dict({"text": [r["text"] for r in rows], "labels": [multihot(r["target"]) for r in rows]})
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
dev_ds = to_ds(dev_rows)

def pos_weight_of(rows):
    Y = np.stack([multihot(r["target"]) for r in rows])
    pos = Y.sum(0); neg = len(Y) - pos
    return np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)

class FocalTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pw = torch.tensor(pos_weight, dtype=torch.float32)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        loss = ((self._pw.to(logits.device) * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def macro_prauc(y, p, n=N_SEL):
    # 앞 n 칸의 PR-AUC 평균 — 양성이 있는 유형만. n=6 이 모델을 고르는 기준이다
    v = [safe(average_precision_score, y[:, i], p[:, i]) for i in range(n)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = 1 / (1 + np.exp(-logits))
    return {"macro_prauc": macro_prauc(labels[SEL_MASK].astype(int), p[SEL_MASK])}

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    th = {}
    for i, l in enumerate(CONF8):
        if y[:, i].sum() < 10:
            th[l] = 0.5
            continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    th[WAITING[0]] = TH_OFF
    return th

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

print("dev 양성 (선택 방식", SCORE_VIEW, "):", {l: int(Y_DEV[SEL_MASK, i].sum()) for i, l in enumerate(LABEL_LIST)})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — CPU 로는 매우 느리다. 런타임 유형을 GPU 로 바꾼다")

## 6. 학습 — 실험 × 시드

🔒 **기준을 먼저 찍고 학습한다.** 시드 하나가 끝나면 dev 확률을 Drive 에 적는다 — 다시 실행하면 끝난 시드는 건너뛴다(`♻️`).
실험 하나가 끝나면 가장 좋은 시드의 **dev 행별 확률 CSV · 문턱 zip · 모델 zip** 을 Drive 에 둔다.

In [ ]:
print("🔒 사전 고정 기준 — 기준 실행", BASE_RUN, "대비 · 선택 방식", SCORE_VIEW, "· 가장 좋은 시드 · 그 시드의 dev 문턱")
print(f"  B   6종 dev macro PR-AUC(시드 평균) 하락이 {GATE['prauc_drop']} 이내")
print(f"  G1  위반 재현율(유형 있는 위반 · 8종 중 하나라도) 하락이 {GATE['recall_drop'] * 100:.0f}%p 이내")
print(f"  G2  유형별 재현율 하락이 {GATE['type_recall_drop'] * 100:.0f}%p 이내 (dev 양성 30건 이상인 유형만)")
print(f"  G3  음성(D · L) 오판정이 +{GATE['neg_flag_up']}줄 이내")
print(f"  채택 후보 = 넷 다 통과 + (PR-AUC 시드 평균 +{GATE['adopt_prauc']} 이상  또는  음성 오판정 −{GATE['adopt_neg_down']}줄 이상)")
print("  ※ dev 에는 해설서 수정문구가 없다 — 「적법 문구를 통과시키는가」는 이 기준으로 재지 못한다. 최종 채택은 팀장 판정이다.\n")

CFG_SIG = [NOTEBOOK, GOLDEN_SHA, DEV_SHA, SCORE_VIEW, MODEL_NAME, MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP, WEIGHT_DECAY, MAX_LEN,
           FOCAL_GAMMA, POS_WEIGHT_CAP, USE_WORD_ROWS, SMOKE]
def sig_of(name, seed):
    ids = hashlib.sha256("|".join(r["id"] for r in TRAIN[name]).encode()).hexdigest()
    return hashlib.sha256(json.dumps(CFG_SIG + [name, seed, ids], ensure_ascii=False).encode()).hexdigest()[:16]

def train_seed(name, seed, get_ds, pw):
    tag = run_tag(name)
    keep, local = f"{OUT_ROOT}/{tag}/seed{seed}", f"./runs/{tag}/seed{seed}"
    done = f"{keep}/done.json"
    if os.path.exists(done) and os.path.exists(f"{keep}/dev_p.npy"):
        res = json.load(open(done, encoding="utf-8"))
        if res.get("sig") == sig_of(name, seed):
            res["dev_p"] = np.load(f"{keep}/dev_p.npy")
            print(f"♻️ {name} · seed {seed}: 이미 끝남 — 건너뜀 (best epoch {res['best_epoch']})")
            return res
        print(f"{name} · seed {seed}: 저장된 기록이 지금 설정과 다르다 — 다시 학습한다")
    t0 = time.time()
    train_ds = get_ds()
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=dict(enumerate(LABEL_LIST)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./runs/{tag}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = FocalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
                           compute_metrics=compute_metrics, pos_weight=pw,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    shutil.rmtree(local, ignore_errors=True)
    trainer.save_model(local)
    shutil.rmtree(f"./runs/{tag}/ckpt{seed}", ignore_errors=True)
    res = {"sig": sig_of(name, seed), "seed": seed, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
           "curve": [[int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)] for h in hist], "minutes": round((time.time() - t0) / 60, 1)}
    os.makedirs(keep, exist_ok=True)
    np.save(f"{keep}/dev_p.npy", dev_p)
    shutil.copy(f"{local}/config.json", f"{keep}/config.json")
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)       # 🔴 done.json 을 맨 끝에 쓴다 — 있으면 이 시드는 끝난 것이다
    res["dev_p"] = dev_p
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

def scheme_of(name, R):
    s, st = R["best_seed"], TRAIN_STATS[name]
    return {
        "label_list": LABEL_LIST, "label_thresholds": R["thresholds"],
        "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": SEL,
        "waiting_note": f"편입 대기(D-321) — 칸은 두되 문턱 {TH_OFF} 로 꺼 둔다",
        "selection_note": "회차 · 시드는 6종 dev macro PR-AUC 로 고른다 (D-321 결정 4)",
        "backbone": MODEL_NAME, "notebook": f"{NOTEBOOK} (박수진 · dev = v11 규칙 · 합성 × 이유 구역 실험)", "experiment": run_tag(name),
        "use_injected": RUN_DEFS[name]["inj"], "use_reason_zone": RUN_DEFS[name]["reason"], "use_word_rows": USE_WORD_ROWS,
        "seed": s, "best_epoch": R["seeds"][s]["best_epoch"], "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS),
        "train_rows": st["학습 행"], "train_injected_rows": st["합성"], "train_reason_rows": st["이유 구역"],
        "train_label_pos": {l: st[l] for l in LABEL_LIST},
        "train_rule": "조건 M 제외 · D · L 음성 · 유형 없는 위반 제외" + ("" if USE_WORD_ROWS else " · 낱말 행 제외")
                      + ("" if RUN_DEFS[name]["inj"] else " · 합성(주입) 행 제외") + ("" if RUN_DEFS[name]["reason"] else " · 의결서 이유 구역 제외"),
        "dev_rule": f"v11 과 같다 — 원천별 묶음 {DEV_FRAC:.0%} + 6종 보충 · 시드 {DEV_SEED} · 문장 단위 · 합성 아님",
        "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "score_view": SCORE_VIEW, "score_rows": int(SEL_MASK.sum()),
        "score_view_note": "v11 = dev 전부(M · 유형 없는 위반 · D · L 음성) / A = 이유 구역 · M · 유형 없는 위반을 뺀 줄",
        "threshold_note": "가장 좋은 시드의 dev F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
        "thresholds_by_seed": {str(k): v["thresholds"] for k, v in R["seeds"].items()},
        "dev_macro_prauc6_by_seed": {str(k): v["prauc6"] for k, v in R["seeds"].items()},
        "final_test": False, "test_note": "이 노트북은 test 를 채점하지 않는다 (D-175)", "smoke": SMOKE,
    }

def export_run(name, R):
    """가장 좋은 시드의 dev 행별 확률 CSV · 문턱 zip · (스위치) 모델 zip 을 결과 폴더에 둔다."""
    tag, s = run_tag(name), R["best_seed"]
    p, scheme = R["seeds"][s]["dev_p"], scheme_of(name, R)
    csv_path = f"{OUT_ROOT}/{tag}_dev행별확률.csv"
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:          # 문구는 넣지 않는다 (id · 갈래 · 확률만)
        w = csv.writer(f)
        w.writerow(["id", "조건", "bucket", "이유구역"] + [f"p_{l}" for l in LABEL_LIST])
        for r, row in zip(dev_rows, p):
            w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r))] + [f"{x:.6f}" for x in row])
    with zipfile.ZipFile(f"{OUT_ROOT}/{tag}_scheme.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("label_scheme.json", json.dumps(scheme, ensure_ascii=False, indent=2))
        z.write(f"{OUT_ROOT}/{tag}/seed{s}/config.json", "config.json")
    out = [os.path.basename(csv_path), f"{tag}_scheme.zip"]
    local = f"./runs/{tag}/seed{s}"
    if not SAVE_MODEL_ZIP:
        pass
    elif not os.path.exists(f"{local}/config.json"):
        if os.path.exists(f"{OUT_ROOT}/{tag}.zip") and R.get("zip_seed") == s:
            out.append(f"{tag}.zip (앞서 저장한 것)")
        else:
            print(f"  🟡 {name}: 가장 좋은 시드 {s} 의 가중치가 이 런타임에 없다(앞 세션에서 학습) — 모델 zip 은 못 만든다. "
                  f"필요하면 Drive 의 {tag} 폴더를 지우고 이 실험만 다시 돌린다")
    else:
        sdir = f"./export/{tag}"
        shutil.rmtree(sdir, ignore_errors=True)
        m = AutoModelForSequenceClassification.from_pretrained(local)
        assert [m.config.id2label[i] for i in range(m.config.num_labels)] == LABEL_LIST, "저장 칸 순서가 다르다 (D-220)"
        m.save_pretrained(sdir, safe_serialization=True)
        tokenizer.save_pretrained(sdir)
        assert os.path.exists(f"{sdir}/tokenizer.json"), "tokenizer.json 이 없다 (앱 로더 필수 파일)"
        json.dump(scheme, open(f"{sdir}/label_scheme.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        try:
            zp = shutil.make_archive(f"./export/{tag}", "zip", sdir)
            shutil.copy(zp, f"{OUT_ROOT}/{tag}.zip")
            json.dump({"zip_seed": s}, open(f"{OUT_ROOT}/{tag}/zip_seed.json", "w"))
            out.append(f"{tag}.zip")
            os.remove(zp); shutil.rmtree(sdir, ignore_errors=True)          # 런타임 디스크 — 결과 폴더에 옮겼으면 지운다
        except Exception as e:
            print(f"  🟡 {name}: 모델 zip 저장 안 됨 (Drive 용량?) — {e}")
        del m
    return out

RESULTS = {}
T_ALL = time.time()
for name in RUNS:
    tag, rows = run_tag(name), TRAIN[name]
    pw = pos_weight_of(rows)
    print(f"\n{'=' * 22} {name} · 학습 {len(rows):,}행 {'=' * 22}")
    print("pos_weight:", {l: round(float(pw[i]), 2) for i, l in enumerate(LABEL_LIST)})
    _cache, seeds = {}, {}
    def get_ds():                                 # 학습할 시드가 있을 때만 토크나이즈한다
        if "ds" not in _cache:
            _cache["ds"] = to_ds(rows)
        return _cache["ds"]
    for seed in SEEDS:
        res = train_seed(name, seed, get_ds, pw)
        y, p = Y_DEV[SEL_MASK], res["dev_p"][SEL_MASK]
        res["prauc6"], res["prauc8"] = macro_prauc(y, p, N_SEL), macro_prauc(y, p, N8)
        res["thresholds"] = pick_thresholds(y, p)
        seeds[seed] = res
        print(f"  seed {seed}: 6종 dev macro PR-AUC {res['prauc6']:.4f} (8종 {res['prauc8']:.4f}) · best epoch {res['best_epoch']}/{res['epochs_run']} · 곡선 {res['curve']} · {res['minutes']}분")
    best = max(SEEDS, key=lambda s: seeds[s]["prauc6"])
    _zs = f"{OUT_ROOT}/{tag}/zip_seed.json"
    R = {"seeds": seeds, "best_seed": best, "thresholds": seeds[best]["thresholds"],
         "prauc6_mean": float(np.mean([seeds[s]["prauc6"] for s in SEEDS])), "prauc6_std": float(np.std([seeds[s]["prauc6"] for s in SEEDS])),
         "zip_seed": json.load(open(_zs))["zip_seed"] if os.path.exists(_zs) else None}
    RESULTS[name] = R
    print(f"[{name}] 6종 dev macro PR-AUC {R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f} · 가장 좋은 시드 {best}")
    print("  문턱:", {l: R["thresholds"][l] for l in CONF8})
    print("  저장:", export_run(name, R))
    for s in SEEDS:                               # 디스크 — 가장 좋은 시드의 가중치만 남긴다
        if s != best:
            shutil.rmtree(f"./runs/{tag}/seed{s}", ignore_errors=True)
print(f"\n학습 끝 — {round((time.time() - T_ALL) / 60, 1)}분")

## 7. 결과 — 네 실험을 같은 dev 로 견준다

- **표 1** 문턱 없이 보는 모델 품질 (PR-AUC). 선택 방식의 시드 평균이 「B」 기준이다.
- **표 2** 문턱을 적용했을 때 갈래별로 얼마나 울리는가 — dev 전체 / 「이유」 구역이 아닌 줄만.
- **표 3** 유형별 재현율 · 정밀도.
- **표 4** 사전 고정 기준 판정.

`dev_probs_v11_…csv` 를 올렸으면 **v11 줄**이 같이 나온다 — v11 은 한 시드의 확률뿐이라 시드 평균 칸이 비고, 기준 판정에서는 빠진다.
v11 줄이 맞게 붙었는지 보는 값 (로컬 대조 10-07): 6종 PR-AUC **0.721** · 위반 탐지 **452/510** · 정답 유형 적중 395/510 · D 12/81 · L 2/5.
읽는 법: 「유형 없는 위반 · 조건 M」 줄이 울리는 것은 틀린 것이 아니다(위반일 수 있는 문구다). **줄어야 하는 것은 D · L** 이다.

In [ ]:
V11_TH = {"질병_예방치료_표방": 0.65, "건강기능식품_오인": 0.575, "의약품_오인": 0.575, "거짓_과장": 0.575, "소비자_기만": 0.45,
          "후기_체험기_기만": 0.5, "부당_비교광고": 0.625, "비방광고": 0.45, "기능성화장품_오인": TH_OFF}   # v11 label_scheme.json 의 값

ENTRIES = {}
for name in RUNS:
    R = RESULTS[name]
    ENTRIES[name] = {"p": R["seeds"][R["best_seed"]]["dev_p"], "th": R["thresholds"], "train_rows": TRAIN_STATS[name]["학습 행"],
                     "seed": f"{R['best_seed']} (epoch {R['seeds'][R['best_seed']]['best_epoch']})",
                     "mean": R["prauc6_mean"], "std": R["prauc6_std"]}
if V11_CSV:
    with open(V11_CSV, encoding="utf-8-sig", newline="") as f:
        _rows = {r["id"]: r for r in csv.DictReader(f)}
    _miss = [r["id"] for r in dev_rows if r["id"] not in _rows]
    if _miss or len(_rows) != len(dev_rows):
        print(f"🟡 v11 확률 파일이 이 dev 와 다른 행이다 (파일 {len(_rows)}줄 · 없는 id {len(_miss)}) — v11 줄을 뺀다")
    else:
        _p = np.array([[float(_rows[r["id"]][f"p_{l}"]) for l in LABEL_LIST] for r in dev_rows], dtype=np.float32)
        ENTRIES["v11 (소성민)"] = {"p": _p, "th": V11_TH, "train_rows": 4110, "seed": "42 (epoch 6)", "mean": None, "std": None}
        _re = pick_thresholds(Y_DEV, _p)
        _same = all(abs(_re[l] - V11_TH[l]) < 1e-9 for l in CONF8)
        print("v11 문턱 대조 — 올린 확률로 같은 규칙(F1 최대)을 다시 돌리면:", "✅ 저장값과 같다" if _same else f"🟡 저장값과 다르다 {_re} — 표는 저장값으로 잰다")

def rate(k, n):
    return "—" if n == 0 else f"{k}/{n} ({k / n * 100:.1f}%)"
def f3(v):
    return "N/A" if v is None or v != v else f"{v:.3f}"

def bucket_counts(e, mask):
    th8 = np.array([e["th"][l] for l in CONF8])
    fl = e["p"][:, :N8] >= th8
    any_, hit = fl.any(1), (fl & (Y_DEV[:, :N8] == 1)).any(1)
    out = {b: (int(any_[mask & (B == b)].sum()), int((mask & (B == b)).sum())) for b in BUCKETS}
    out["hit"] = (int(hit[mask & (B == "scored")].sum()), out["scored"][1])
    _neg = mask & np.isin(B, ["D", "L", "neg"])
    out["음성"] = (int(any_[_neg].sum()), int(_neg.sum()))
    i = LABEL2ID["거짓_과장"]
    out["거짓"] = (int(fl[mask, i].sum()), int(mask.sum()))
    out["거짓_오발"] = (int((fl[:, i] & (Y_DEV[:, i] == 0))[mask].sum()), int(((Y_DEV[:, i] == 0) & mask).sum()))
    return out

def type_stats(e, mask):
    out = {}
    for i, l in enumerate(CONF8):
        y, f = Y_DEV[mask, i], e["p"][mask, i] >= e["th"][l]
        n, tp, pp = int(y.sum()), int((f & (y == 1)).sum()), int(f.sum())
        out[l] = {"n": n, "tp": tp, "pp": pp, "R": tp / n if n else None, "P": tp / pp if pp else None,
                  "prauc": safe(average_precision_score, y, e["p"][mask, i])}
    return out

# ── 표 1 · 모델 품질 (문턱 없음)
_t = {}
for k, e in ENTRIES.items():
    _t[k] = {"학습 행": f"{e['train_rows']:,}", "선택 시드": e["seed"],
             f"6종 PR-AUC 시드 평균 ({SCORE_VIEW} 방식)": "— (한 시드)" if e["mean"] is None else f"{e['mean']:.4f} ± {e['std']:.4f}",
             "선택 시드 6종 PR-AUC · v11 방식": f3(macro_prauc(Y_DEV[VIEW["v11"]], e["p"][VIEW["v11"]])),
             "· A 방식": f3(macro_prauc(Y_DEV[VIEW["A"]], e["p"][VIEW["A"]])),
             "· 「이유」 구역 줄만": f3(macro_prauc(Y_DEV[IS_REASON], e["p"][IS_REASON])),
             "8종 PR-AUC · v11 방식": f3(macro_prauc(Y_DEV, e["p"], N8))}
print(f"■ 표 1 · 모델 품질 — dev {len(dev_rows)}줄 (v11 방식 {int(VIEW['v11'].sum())} · A 방식 {int(VIEW['A'].sum())} · 「이유」 구역 {int(IS_REASON.sum())})")
display(pd.DataFrame(_t).T)

# ── 표 2 · 문턱 적용 — 갈래별로 8종 중 하나라도 울린 줄
for title, mask in [("dev 전체", VIEW["v11"]), ("「이유」 구역이 아닌 줄만", ~IS_REASON)]:
    _t = {}
    for k, e in ENTRIES.items():
        c = bucket_counts(e, mask)
        _t[k] = {"위반 탐지 (하나라도) ↑": rate(*c["scored"]), "정답 유형 적중 ↑": rate(*c["hit"]),
                 "유형 없는 위반": rate(*c["pending"]), "조건 M": rate(*c["M"]),
                 "D 오판정 ↓": rate(*c["D"]), "L 오판정 ↓": rate(*c["L"]),
                 "거짓_과장이 울린 줄": rate(*c["거짓"]), "정답이 거짓_과장 아닌 줄에서 ↓": rate(*c["거짓_오발"])}
    print(f"\n■ 표 2 · 문턱 적용 ({title} · {int(mask.sum())}줄) — 선택 시드 · 그 모델의 문턱")
    display(pd.DataFrame(_t).T)

# ── 표 3 · 유형별 (선택 방식의 행)
_ts = {k: type_stats(e, SEL_MASK) for k, e in ENTRIES.items()}
_t = {}
for l in CONF8:
    n = _ts[BASE_RUN][l]["n"]
    _t[l] = {"dev 양성": f"{n}" + (" (측정 불가 n<30)" if n < 30 else "")}
    for k in ENTRIES:
        s = _ts[k][l]
        _t[l][k] = f"R {f3(s['R'])} · P {f3(s['P'])} · AP {f3(s['prauc'])} · τ {ENTRIES[k]['th'][l]}"
print(f"\n■ 표 3 · 유형별 재현율 R · 정밀도 P · PR-AUC(AP) · 문턱 τ — {SCORE_VIEW} 방식의 행 {int(SEL_MASK.sum())}줄")
display(pd.DataFrame(_t).T)

# ── 표 4 · 사전 고정 기준 (기준 실행 대비)
def gate_view(k):
    e, c, t = ENTRIES[k], bucket_counts(ENTRIES[k], SEL_MASK), _ts[k]
    return {"mean": e["mean"], "rec": c["scored"][0] / max(c["scored"][1], 1), "rec_k": c["scored"], "neg": c["음성"][0], "neg_n": c["음성"][1],
            "type": {l: t[l]["R"] for l in CONF8 if _ts[BASE_RUN][l]["n"] >= 30}}
_b = gate_view(BASE_RUN)
VERDICT, _t = {}, {}
for k in RUNS:
    g = gate_view(k)
    if k == BASE_RUN:
        _t[k] = {"PR-AUC 시드 평균 (B)": f"{g['mean']:.4f}", "위반 재현율 (G1)": rate(*g["rec_k"]), "유형별 최대 하락 (G2)": "—",
                 "음성 오판정 (G3)": f"{g['neg']}/{g['neg_n']}", "판정": "기준"}
        continue
    d_pr, d_rec, d_neg = g["mean"] - _b["mean"], g["rec"] - _b["rec"], g["neg"] - _b["neg"]
    d_type = {l: g["type"][l] - _b["type"][l] for l in g["type"]}
    worst = min(d_type, key=d_type.get) if d_type else None
    ok = {"B": d_pr >= -GATE["prauc_drop"], "G1": d_rec >= -GATE["recall_drop"],
          "G2": (d_type[worst] >= -GATE["type_recall_drop"]) if worst else True, "G3": d_neg <= GATE["neg_flag_up"]}
    fails = [n for n, v in ok.items() if not v]
    if fails:
        verdict = "탈락 (" + " · ".join(fails) + ")"
    elif d_pr >= GATE["adopt_prauc"] or d_neg <= -GATE["adopt_neg_down"]:
        verdict = "채택 후보"
    else:
        verdict = "통과 · 기준과 차이 없음"
    VERDICT[k] = {"verdict": verdict, "d_prauc": d_pr, "d_recall": d_rec, "d_neg": d_neg, "worst_type": worst, "d_worst": d_type.get(worst)}
    _t[k] = {"PR-AUC 시드 평균 (B)": f"{g['mean']:.4f} ({d_pr:+.4f})", "위반 재현율 (G1)": f"{rate(*g['rec_k'])} ({d_rec * 100:+.1f}%p)",
             "유형별 최대 하락 (G2)": "—" if worst is None else f"{worst} {d_type[worst] * 100:+.1f}%p",
             "음성 오판정 (G3)": f"{g['neg']}/{g['neg_n']} ({d_neg:+d}줄)", "판정": verdict}
print(f"\n■ 표 4 · 사전 고정 기준 — 기준 {BASE_RUN} 대비 · {SCORE_VIEW} 방식의 행")
display(pd.DataFrame(_t).T)
print("※ 판정은 dev 기준이다. 채택 후보가 나와도 test 채점은 팀장 판정 뒤 1회다 (D-175). dev 의 D 는 주의사항 문구 위주라 「수정문구 통과」는 여기서 보이지 않는다.")
if SMOKE:
    print("⚠️ SMOKE 실행 — 위 수치는 의미 없다")

SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "score_view": SCORE_VIEW,
           "gate": GATE, "base_run": BASE_RUN, "smoke": SMOKE, "train": TRAIN_STATS, "verdict": VERDICT,
           "runs": {k: {"best_seed": RESULTS[k]["best_seed"], "prauc6_mean": RESULTS[k]["prauc6_mean"], "prauc6_std": RESULTS[k]["prauc6_std"],
                        "thresholds": RESULTS[k]["thresholds"],
                        "seeds": {str(s): {x: v[x] for x in ("prauc6", "prauc8", "best_epoch", "epochs_run", "curve", "minutes", "thresholds")}
                                  for s, v in RESULTS[k]["seeds"].items()},
                        "buckets_all": bucket_counts(ENTRIES[k], VIEW["v11"]), "buckets_nonreason": bucket_counts(ENTRIES[k], ~IS_REASON),
                        "types": _ts[k]} for k in RUNS}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_v10.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("\n결과 폴더:", OUT_ROOT)
for f in sorted(os.listdir(OUT_ROOT)):
    if os.path.isfile(f"{OUT_ROOT}/{f}"):
        print(f"  {f:70s} {os.path.getsize(f'{OUT_ROOT}/{f}') / 1e6:8.1f} MB")

## 8. 다음 단계

**1) 판정 로직에 붙여 보기 (로컬 · dev)** — 실험마다 Drive 의 두 파일을 받아 저장소 밖(`C:\Users\<나>\copylane_local\`)에 둔다.

```
uv run python docs\psj\e2e_prototype\run_judge_dist.py --model <…_scheme.zip> --probs-csv <…_dev행별확률.csv> --split dev
```

`--conditional` 을 붙이면 품목을 아는 조건이다. 스크립트가 「dev 규칙(v8 ~ v8.2)과 다른 행」이라고 알리는 것은 정상이다 — 이 dev 는 v11 규칙이다.

**2) 무엇을 보고 고르나**

- 표 4 가 「채택 후보」 → 그 실험과 기준을 판정 로직 dev 결과로 한 번 더 견준다.
- 전부 「차이 없음」 → 기준(= A 와 같은 재료)을 유지한다. 재료를 빼도 dev 가 같다는 것 자체가 기록할 결과다.
- 어느 경우든 **test 는 팀장 판정 뒤 1회** 다. 이 노트북의 결과로 test 를 다시 재지 않는다.

**3) 올리지 않는 것** — 모델 zip · 행별 확률 CSV · 이 노트북의 실행 출력은 저장소에 올리지 않는다 (D-249). 노트북은 출력을 지우고 올린다.